In [ ]:
%%capture
!pip install wandb
!apt-get install git
!apt autoremove
!pip3 install awscli

!mkdir -p /root/workspace/data/
!mkdir -p /root/workspace/out/

In [ ]:
%%capture
%cd /root/workspace
!git clone https://github.com/chaitjo/geometric-gnn-dojo.git
!pip3 install -r /root/workspace/UnitSphere/requirements.txt

In [ ]:
%cd /root/workspace/geometric-gnn-dojo/
!git stash
!git pull

In [ ]:
# %%capture
%cd /root/workspace
!cp /root/workspace/UnitSphere/ext/train_nll_utils.py ./geometric-gnn-dojo/experiments/utils/train_utils.py # remove once iclr is pulled
!cp /root/workspace/UnitSphere/ext/comenet.py ./geometric-gnn-dojo/models/ # remove once iclr is pulled
!echo "from models.comenet import ComENetModel" >> ./geometric-gnn-dojo/models/__init__.py

# Models

# Datasets

In [ ]:
from abc import ABCMeta
import ast

import time
import torch
from typing import Tuple
from torch import Tensor
from torch_sparse import SparseTensor
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import sys

sys.path.append('/root/workspace/alignment/pyorbit/utils/')
from scipy.spatial import ConvexHull

def triplets(
    edge_index: Tensor,
    num_nodes: int,
) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor, Tensor, Tensor]:
    row, col = edge_index  # j->i

    value = torch.arange(row.size(0), device=row.device)
    adj_t = SparseTensor(row=col, col=row, value=value,
                         sparse_sizes=(num_nodes, num_nodes))
    adj_t_row = adj_t[row]
    num_triplets = adj_t_row.set_value(None).sum(dim=1).to(torch.long)

    # Node indices (k->j->i) for triplets.
    idx_i = col.repeat_interleave(num_triplets)
    idx_j = row.repeat_interleave(num_triplets)
    idx_k = adj_t_row.storage.col()
    mask = idx_i != idx_k  # Remove i == k triplets.
    idx_i, idx_j, idx_k = idx_i[mask], idx_j[mask], idx_k[mask]

    # Edge indices (k-j, j->i) for triplets.
    idx_kj = adj_t_row.storage.value()[mask]
    idx_ji = adj_t_row.storage.row()[mask]

    return col, row, idx_i, idx_j, idx_k, idx_kj, idx_ji

class Frame(metaclass=ABCMeta):
    def __init__(self, tol=1e-2, *args, **kwargs):
        super().__init__()
        self.tol = tol

    def get_frame(self, data, *args, **kwargs):

        # TRANSLATION INVARIANCE
        data = self.check_type(data) # Assert Type
        data = data - np.mean(data, axis=0) # Assert Centered
        data = data[np.linalg.norm(data, axis=1) > self.tol]

        # PROJECT ONTO SPHERE
        distances = np.linalg.norm(data, axis=1, keepdims=False)
        shell_data =  data/np.linalg.norm(data, axis=1, keepdims=True)
        data_duplicate_dict = {}
        for i, d in enumerate(shell_data):
          for j in range(i+1, len(shell_data)):
            if np.allclose(d, shell_data[j]):
              if i not in data_duplicate_dict:
                data_duplicate_dict[i] = [j]
              else:
                data_duplicate_dict[i] += [j]

        # FIND NEAREST TWO NEIGHBORS
        edges = []
        hull = ConvexHull(shell_data, qhull_options='Qx')
        for simplex in hull.simplices:
          edges.append(simplex)
          if simplex[0] in data_duplicate_dict:
            for j in data_duplicate_dict[simplex[0]]:
              edges.append([j, simplex[1]])
              # edges.append([simplex[0], j])
          if simplex[1] in data_duplicate_dict:
            for j in data_duplicate_dict[simplex[1]]:
              edges.append([simplex[0], j])
              # edges.append([j, simplex[1]])

        edge_index = np.array(list(edges))
        edge_index = torch.from_numpy(edge_index).T.to(torch.long)
        edge_index = to_undirected(edge_index)
        # Generate edge attributes
        print(distances.shape)
        edge_attr = torch.from_numpy(distances).reshape(-1,1)

        # DIMENET WAY
        i, j, idx_i, idx_j, idx_k, idx_kj, idx_ji = triplets(
            edge_index,
            num_nodes=shell_data.shape[0])

        # Calculate distances.
        shell_data = torch.from_numpy(shell_data)
        dist = (shell_data[i] - shell_data[j]).pow(2).sum(dim=-1).sqrt()
        # Calculate angles.
        pos_jk, pos_ij = shell_data[idx_j] - shell_data[idx_k], shell_data[idx_i] - shell_data[idx_j]
        a = (pos_ij * pos_jk).sum(dim=-1)
        b = pos_ij[:, 0] * pos_jk[:, 1] - pos_ij[:, 1] * pos_jk[:, 0] #torch.cross(pos_ij, pos_jk, dim=1).norm(dim=-1)
        angle = torch.atan2(b, a)
        angle_attr = torch.zeros(shell_data.shape[0], shell_data.shape[0])
        for i,index in enumerate(idx_kj):
          edge = edge_index[:,index]
          angle_attr[edge[0], edge[1]] = angle[i]
          angle_attr[edge[1], edge[0]] = angle[i]
        # make 8xN matrix filled in with angles or zeros elsewhere
        edge_attr = torch.cat([edge_attr, angle_attr], dim=1)

        return edge_index, edge_attr

    def check_type(self, data, *args, **kwargs):
        if isinstance(data, torch.Tensor):
            return data.detach().cpu().numpy()
        elif isinstance(data, np.ndarray):
            return data
        else:
            raise TypeError(f"Data type not supported {type(data)}")
# dataset = create_kchains(k=6, connectivity='convhull')
# for data in dataset:
#     plot_3d(data, lim=2*k)

In [ ]:
from scipy.spatial import Delaunay, delaunay_plot_2d
from scipy.spatial import Voronoi, voronoi_plot_2d


def compute_convhull_edges(pos, vis=False):
        edges = []
        hull = ConvexHull(pos, qhull_options='Qx')
        for simplex in hull.simplices:
          edges.append(simplex)
        edge_index = np.array(list(edges))
        edge_index = torch.from_numpy(edge_index).T.to(torch.long)
        edge_index = to_undirected(edge_index)
        return edge_index


def compute_voronoi_edges(pos, vis=False):
    pos_np = pos.numpy()  # Convert to numpy array
    tri = Delaunay(pos)
    vor = Voronoi(pos_np)
    if vis:
      try:
        voronoi_plot_2d(vor)
        delaunay_plot_2d(tri)
      except:
        pass
    rows, cols = tri.vertex_neighbor_vertices
    edges = []
    for i in range(len(rows) - 1):
        start, end = rows[i], rows[i + 1]
        neighbors = cols[start:end]
        for neighbor in neighbors:
            edges.append([i, neighbor])

    return edges


## Simple Chain Dataset

In [ ]:
import sys
sys.path.append('/root/workspace/geometric-gnn-dojo/')

import torch
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import KNNGraph
from torch_geometric.utils import to_undirected
import e3nn
from functools import partial

from torch_geometric.seed import seed_everything

from experiments.utils.plot_utils import plot_3d

def create_kchains(k,connectivity='radius'):
    seed_everything(10)
    assert k >= 2
    assert connectivity in ['radius', 'knn', 'voronoi', 'convhull', 'full', 'unitsphere']

    dataset = []

    # Graph 0
    atoms = torch.LongTensor( [0] + [0] + [0]*(k-1) + [0] )
    cell = torch.diag(torch.ones(3,dtype=torch.float)).view(1,3,3)
    edge_index = torch.LongTensor( [ [i for i in range((k+2) - 1)], [i for i in range(1, k+2)] ] )
    pos = torch.FloatTensor(
        [[-4, -3, 0]] +
        [[0, 5*i , 0] for i in range(k)] +
        [[4, 5*(k-1) + 3, 0]]
    )
    y = torch.LongTensor([0])  # Label gvp0
    data1 = Data(atoms=atoms, edge_index=edge_index, pos=pos, y=y, natoms=k+2, cell=cell)

    # Edges
    if connectivity == 'voronoi':
      voronoi_edges = compute_voronoi_edges(data1.pos[:,:-1])
      data1.edge_index = torch.tensor(voronoi_edges, dtype=torch.long).t().contiguous()
    elif connectivity == 'convhull':
      data1.edge_index = compute_convhull_edges(data1.pos[:,:-1])
    elif connectivity == 'unitsphere':
      frame = Frame()
      data1.edge_index, data1.edge_attr =  frame.get_frame(data1.pos[:,:-1])
    elif connectivity == 'knn':
      data1 = KNNGraph(3)(data1)
    elif connectivity == 'full':
      edge_index = []
      for i in range(k+2):
        for j in range(k+2):
          edge_index.append([i,j])
          edge_index.append([j,i])
      data1.edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    edge_index = to_undirected(data1.edge_index)
    edges_set = set(map(tuple, edge_index.t().tolist()))
    data1.edge_index = torch.tensor(list(edges_set), dtype=torch.long).t()

    dataset.append(data1)

    # Graph 1
    atoms = torch.LongTensor( [0] + [0] + [0]*(k-1) + [0] )
    edge_index = torch.LongTensor( [ [i for i in range((k+2) - 1)], [i for i in range(1, k+2)] ] )
    pos = torch.FloatTensor(
        [[4, -3, 0]] +
        [[0, 5*i , 0] for i in range(k)] +
        [[4, 5*(k-1) + 3, 0]]
    )
    y = torch.LongTensor([1])  # Label 1
    data2 = Data(atoms=atoms, edge_index=edge_index, pos=pos, y=y, natoms=k+2, cell=cell)

    # Edges
    if connectivity == 'voronoi':
      voronoi_edges = compute_voronoi_edges(data2.pos[:,:-1])
      data2.edge_index = torch.tensor(voronoi_edges, dtype=torch.long).t().contiguous()
    elif connectivity == 'unitsphere':
      frame = Frame()
      data2.edge_index, data2.edge_attr =  frame.get_frame(data2.pos[:,:-1])
    elif connectivity == 'convhull':
      data2.edge_index = compute_convhull_edges(data2.pos[:,:-1])
    elif connectivity == 'knn':
      data2 = KNNGraph(3)(data2)
    elif connectivity == 'full':
      edge_index = []
      for i in range(k+2):
        for j in range(k+2):
          edge_index.append([i,j])
          edge_index.append([j,i])
      data2.edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    edge_index = to_undirected(data2.edge_index)
    edges_set = set(map(tuple, edge_index.t().tolist()))
    data2.edge_index = torch.tensor(list(edges_set), dtype=torch.long).t()

    dataset.append(data2)

    return dataset

# Create dataset
for connectivity in ['radius','knn','convhull','voronoi','full','unitsphere']:
  k = 6
  print(f'Connectivity: {connectivity}')
  dataset = create_kchains(k=k, connectivity=connectivity)
  for data in dataset:
      print(data.edge_index)
      plot_3d(data, lim=2*k)

# Experiments

## Simple Chain Experiment

In [ ]:
# Create dataloaders
import random

from experiments.utils.train_utils import run_experiment
from models import SchNetModel, DimeNetPPModel, SphereNetModel, ComENetModel

def run(model_name,cutoff_name=None):
  k = 6
  num_layers = 1
  for connectivity in ['radius','knn','convhull','voronoi','full','unitsphere']:
  # for connectivity in ['full', 'unitsphere']:
      print('*'*20 + f'\nConnectivity: {connectivity}\n' + '*'*20)
      dataset = create_kchains(k=k, connectivity=connectivity)
      for cutoff in range(1,11):
      # for cutoff in [8]:
        print(f"\nCutoff: {cutoff}")
        print(f"Chain Length: {k}")


        # Create dataloaders
        dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
        val_loader = DataLoader(dataset, batch_size=2, shuffle=False)
        test_loader = DataLoader(dataset, batch_size=2, shuffle=False)


        # use_edge_attr = True if connectivity == 'convhull' else False
        # use_edge_attr = False

        correlation = 2
        kwargs = {cutoff_name:cutoff} if cutoff_name else {}
        model = {
            # INV
            "schnet": partial(SchNetModel,  num_gaussians=64, num_filters=64, pool='mean'),
            "dimenet": DimeNetPPModel,
            "spherenet": partial(SphereNetModel, out_emb_channels=256),
            "comenet": partial(ComENetModel, hidden_channels=128, num_radial=8, num_spherical=8),
        }[model_name](num_layers=num_layers, in_dim=1, out_dim=2, **kwargs)

        best_val_acc, test_acc, train_time = run_experiment(
            model,
            dataloader,
            val_loader,
            test_loader,
            n_epochs=100,
            n_times=10,
            verbose=False,
            device='cuda',
        )


In [ ]:
# SCHNET
run('schnet','cutoff')

In [ ]:
# DIMENET
run('dimenet','cutoff')

In [ ]:
# SPHERENET
run('spherenet','cutoff')

In [ ]:
# COMENET
run('comenet','cutoff')